# LLM-Powered SOC Copilot — Experiment Notebook

Retrieval-augmented triage for a security operations center:

**Alert in → RAG (MITRE ATT&CK + similar past alerts) → triage summary /
severity justification / recommended response → human-in-the-loop approve/edit/reject → metrics.**

This notebook mirrors `scripts/run_pipeline.py` end to end. It runs fully
offline: retrieval is TF-IDF + numpy cosine similarity, and the "LLM" is a
deterministic template fallback. To use a real local model, pass a
`LLMInterface(provider="ollama")` — the call slot is `rag.LLMInterface.generate`.

## 0. Setup (imports + repo path)

In [ ]:
import os, sys, time
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../src"))

from soccopilot.alert_store import AlertStore, generate_alerts
from soccopilot.knowledge import load_techniques, severity_name
from soccopilot.rag import TechniqueRetriever, AlertRetriever, LLMInterface
from soccopilot.triage import TriageEngine
from soccopilot.human_loop import run_human_loop
from soccopilot.evaluate import compute_metrics, write_metrics, plot_figures

SEED = 7

## 1. Synthetic alert stream

300 alerts templated from the 20 bundled MITRE ATT&CK techniques. Each row has
an evidence `raw_log`, a ground-truth technique, a 15% decoy (false-positive)
rate and a simulated **analyst verdict** (ground truth + ~5% human error).

In [ ]:
alerts = generate_alerts(n=300, seed=SEED)
alerts.head()
alerts["true_label"].value_counts()

## 2. SQLite alert store + train/test split

In [ ]:
store = AlertStore("../results/soc_alerts.db")
store.reset()
store.save_alerts(alerts)

from sklearn.model_selection import train_test_split
train, test = train_test_split(alerts, test_size=0.30, random_state=SEED,
                               stratify=alerts["analyst_verdict"])
train, test = train.reset_index(drop=True), test.reset_index(drop=True)
print(f"train={len(train)} test={len(test)}")

## 3. Retrieval (RAG over ATT&CK + past alerts)

`TechniqueRetriever` embeds the technique corpus with TF-IDF and ranks by cosine
similarity. `AlertRetriever` does the same over already-triaged alerts so the
copilot can cite "similar past alerts".

In [ ]:
techniques = load_techniques()
tech_retriever = TechniqueRetriever(techniques)
alert_retriever = AlertRetriever(train[["id", "raw_log", "analyst_verdict"]])

sample = test.iloc[0]
print("raw log:", sample["raw_log"][:110], "...")
print("ground truth:", sample["technique_ground_truth"])
for t, s in tech_retriever.retrieve(sample["raw_log"], top_k=3):
    print(f"  technique {t.id} {t.name:<38} sim={s:.3f}")
print("similar past alerts:", alert_retriever.retrieve(sample["raw_log"], top_k=2))

## 4. Fit the retrieval-augmented triage classifier

Features = TF-IDF of the raw log + retrieval-aware numerics
(severity, top-1 technique similarity, technique baseline severity). Labels are
the **analyst verdicts** from the training split (copilot learns from past human
reviews).

In [ ]:
engine = TriageEngine(techniques, tech_retriever)
engine.set_past_alerts(train[["id", "raw_log", "analyst_verdict"]])
engine.fit(train)
print("classes:", engine.model.classes_)

## 5. Triage one alert — see the generated summary

This is where the optional real LLM would be called
(`LLMInterface.generate`); by default the deterministic retrieval+template
fallback runs.

In [ ]:
ai = engine.triage_alert(sample)
print("AI label:", ai["ai_label"], f"(conf={ai['ai_conf']:.3f})")
print("Technique:", ai["ai_technique"], f"(sim={ai['ai_technique_conf']:.3f})")
print("---- summary ----")
print(ai["summary"])
print("---- justification ----")
print(ai["severity_justification"])

## 6. Human-in-the-loop: approve / edit / reject

Decision rule — AI agrees with the analyst → **approved**; AI false positive →
**rejected**; AI misfired but real → **edited**. Assisted analyst time is a
fraction of baseline depending on the action.

In [ ]:
from soccopilot.human_loop import simulate_action, analyst_time
for lbl in ["benign", "suspicious", "malicious"]:
    for ai in ["benign", "suspicious", "malicious"]:
        a = simulate_action(ai, lbl)
        print(f"AI={ai:<11} analyst={lbl:<11} -> {a:<9} time={analyst_time(a, 12.0):.2f} min")

## 7. Full evaluation on the held-out test set

In [ ]:
result_rows = []
for _, alert in alerts.iterrows():
    ai = engine.triage_alert(alert)
    outcome = run_human_loop(store, alert, ai)
    row = alert.to_dict()
    row.update({"ai_label": ai["ai_label"], "ai_conf": ai["ai_conf"],
                "ai_technique": ai["ai_technique"],
                "ai_technique_conf": ai["ai_technique_conf"],
                "summary": ai["summary"],
                "severity_justification": ai["severity_justification"],
                "recommended_steps": ai["recommended_steps"],
                "action": outcome["action"],
                "assisted_time": outcome["assisted_time"],
                "is_test": int(alert["id"] in set(test["id"]))})
    result_rows.append(row)

final = pd.DataFrame(result_rows)
actions = store.get_actions()
metrics = compute_metrics(final, actions, tech_retriever, 0.0, len(train), len(test))
metrics

In [ ]:
write_metrics(metrics, "../results/metrics.md")
print(plot_figures(metrics, final, "../results/figures/"))

print(f"triage accuracy vs analyst labels : {metrics['triage_accuracy']:.3f}")
print(f"ATT&CK mapping top-1 / top-3      : {metrics['map_top1']:.3f} / {metrics['map_top3']:.3f}")
print(f"response-time reduction           : {metrics['time_reduction']*100:.1f}%")
print(f"actions approved/edited/rejected  : {metrics['approved']}/{metrics['edited']}/{metrics['rejected']}")

## Where the real LLM plugs in

`LLMInterface(provider="ollama").generate(context)` is the single call site.
Uncomment and run with a local Ollama server (`ollama pull llama3`) to replace
the template fallback:

```python
# llm = LLMInterface(provider="ollama", model="llama3")
# print(llm.available(), "->", llm.generate(context)[:300])
```

The pipeline degrades gracefully: if Ollama/OpenAI is unreachable it silently
uses the offline fallback, so this notebook always completes.